## Library

In [1]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd

### 1.Verificação dos diretorios

In [2]:
# Verificação dos diretorios
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
     PROJECT_ROOT = PROJECT_ROOT.parent

NOTEBOOK_DIR  = Path.cwd()
DATA_DIR      = PROJECT_ROOT / "data"
BRONZE_DIR    = DATA_DIR / "bronze"
STAGING_DIR   = DATA_DIR / "staging"
SILVER_DIR    = DATA_DIR / "silver"
GOLD_DIR      = DATA_DIR / "gold"    

print(f"Projeto  : {PROJECT_ROOT}")
print(f"Notebook : {NOTEBOOK_DIR}")
print(f"Bronze   : {BRONZE_DIR}")
print(f"Staging  : {STAGING_DIR}")
print(f"Silver   : {SILVER_DIR}")
print(f"Gold     : {GOLD_DIR}")

Projeto  : /home/akel/PycharmProjects/ENEM
Notebook : /home/akel/PycharmProjects/ENEM/notebook
Bronze   : /home/akel/PycharmProjects/ENEM/data/bronze
Staging  : /home/akel/PycharmProjects/ENEM/data/staging
Silver   : /home/akel/PycharmProjects/ENEM/data/silver
Gold     : /home/akel/PycharmProjects/ENEM/data/gold


### 2. Verificar os ZIPs disponíveis

In [3]:
arquivos_zip = sorted(BRONZE_DIR.glob("microdados_enem_*.zip"))

print(f"Total de arquivos ZIP: {len(arquivos_zip)}")

# arquivos_zip[0].name
# # listar os arquivos
# for arquivo in arquivos_zip:
#     print(arquivo.name)


Total de arquivos ZIP: 28


#### 2.1 Verificar anos ausentes

In [4]:
anos_esperados = set(range(1998, 2026))

# arquivos_zip[0].stem.split("_")[-1] --> pegar o ano
anos_encontrados = {
    int(arquivo.stem.split("_")[-1])
    for arquivo in arquivos_zip
}

anos_ausentes = sorted(anos_esperados - anos_encontrados)
anos_inesperados = sorted(anos_encontrados - anos_esperados)

print("Anos ausentes:", anos_ausentes)
print("Anos inesperados:", anos_inesperados)

Anos ausentes: []
Anos inesperados: []


### 3.Testes
#### 3.1 leitura do arquivo

In [5]:
ano_teste = '1998'

zip_teste = BRONZE_DIR / f"microdados_enem_{ano_teste}.zip"

if not zip_teste.exists():
    raise FileNotFoundError(f"Arquivo NÃO encontrado: {zip_teste}")

print(zip_teste)
print(f"Tamanho: {zip_teste.stat().st_size / 1024**3:.2f} GB")

/home/akel/PycharmProjects/ENEM/data/bronze/microdados_enem_1998.zip
Tamanho: 0.02 GB


#### 3.2 Verificar arquivos do ZIP

In [6]:
with ZipFile(zip_teste, mode="r") as arquivo_zip:
    informacoes = arquivo_zip.infolist()

conteudo = pd.DataFrame(
    {
        "caminho_interno": [info.filename for info in informacoes],
        "tamanho_mb": [info.file_size / 1024**2 for info in informacoes],
        "compactado_mb": [info.compress_size / 1024**2 for info in informacoes],
        "diretorio": [info.is_dir() for info in informacoes],
    }
)

conteudo["taxa_compressao"] = (
    1 - conteudo["compactado_mb"] / conteudo["tamanho_mb"]
).where(conteudo["tamanho_mb"] > 0)

conteudo.sort_values(
    by="tamanho_mb",
    ascending=False,
).reset_index(drop=True)

,caminho_interno,tamanho_mb,compactado_mb,diretorio,taxa_compressao
0,DADOS/MICRODADOS_ENEM_1998.csv,60.144619,10.888841,False,0.818956
1,LEIA-ME E DOCUMENTOS TÉCNICOS/entenda_a_sua_no...,11.927711,11.659635,False,0.022475
2,PROVAS E GABARITOS/ENEM_1998_PROVA_AMARELA.pdf,3.168825,0.261564,False,0.917457
3,LEIA-ME E DOCUMENTOS TÉCNICOS/enem_procediment...,0.717774,0.546204,False,0.239032
4,LEIA-ME E DOCUMENTOS TÉCNICOS/Relatorio_pedago...,0.285675,0.216043,False,0.243747
5,LEIA-ME E DOCUMENTOS TÉCNICOS/Manual_Inscrito_...,0.195721,0.188235,False,0.038245
6,LEIA-ME E DOCUMENTOS TÉCNICOS/Leia_Me_Enem_199...,0.165162,0.146112,False,0.115339
7,DICION╡RIO/Dicionário_Microdados_ENEM_1998.xlsx,0.042477,0.039624,False,0.067153
8,INPUTS/INPUT_SAS_MICRODADOS_ENEM_1998.sas,0.041908,0.008062,False,0.807619
9,INPUTS/INPUT_R_MICRODADOS_ENEM_1998.R,0.039230,0.004288,False,0.890704


#### 3.3  selecionar arquivo CSV

In [7]:
arquivos_csv = conteudo[
    conteudo["caminho_interno"]
    .str.lower()
    .str.endswith(".csv")
].copy()
arquivos_csv 

,caminho_interno,tamanho_mb,compactado_mb,diretorio,taxa_compressao
17,DADOS/MICRODADOS_ENEM_1998.csv,60.144619,10.888841,False,0.818956


In [8]:
# selecionando .csv
arquivos_csv = conteudo[
    conteudo["caminho_interno"]
    .str.lower()
    .str.endswith(".csv")
].copy()




if arquivos_csv.empty:
    raise FileNotFoundError(
        f"Nenhum CSV encontrado dentro de {zip_teste.name}"
    )

# Escolhendo o maior arquivo 
csv_principal = (arquivos_csv.sort_values("tamanho_mb", ascending=False).iloc[0])

print("CSV principal:")
print(csv_principal["caminho_interno"])
print(f'Tamanho descompactado: {csv_principal["tamanho_mb"]:.2f} MB')

CSV principal:
DADOS/MICRODADOS_ENEM_1998.csv
Tamanho descompactado: 60.14 MB


In [9]:
conteudo[
    conteudo["caminho_interno"]
    .str.lower()
    .str.endswith(".csv")
].copy()

,caminho_interno,tamanho_mb,compactado_mb,diretorio,taxa_compressao
17,DADOS/MICRODADOS_ENEM_1998.csv,60.144619,10.888841,False,0.818956
